# PET REMOTE SYSTEM_(2) DATA PROCESSING

In [9]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

def load_pet_json_data(data_path):
    """
    로컬 환경에서 JSON 파일들을 로드하는 함수
    """
    print(f"🔍 데이터 로딩 시작: {data_path}")
    
    # JSON 파일 목록 가져오기
    json_files = [f for f in os.listdir(data_path) if f.endswith('.json')]
    print(f"📁 발견된 JSON 파일: {len(json_files)}개")
    
    if not json_files:
        print("❌ JSON 파일을 찾을 수 없습니다!")
        return None
    
    # 첫 번째 파일로 구조 확인
    sample_file = json_files[0]
    print(f"📋 샘플 파일 구조 확인: {sample_file}")
    
    with open(os.path.join(data_path, sample_file), 'r', encoding='utf-8') as f:
        sample_data = json.load(f)
    
    print("🔍 샘플 데이터 구조:")
    if isinstance(sample_data, dict):
        print(f"   - 키 개수: {len(sample_data.keys())}")
        print(f"   - 주요 키들: {list(sample_data.keys())[:10]}")
    
    # 모든 JSON 파일 로드
    data_list = []
    
    for i, filename in enumerate(json_files):
        if i % 100 == 0:  # 100개마다 진행상황 출력
            print(f"   진행중... {i+1}/{len(json_files)}")
        
        try:
            with open(os.path.join(data_path, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)
                # 파일명을 ID로 추가
                data['pet_id'] = filename.replace('.json', '')
                data_list.append(data)
                
        except Exception as e:
            print(f"❌ {filename} 로드 실패: {e}")
    
    print(f"✅ 총 {len(data_list)}개 파일 로드 완료")
    
    # DataFrame 생성
    df = pd.DataFrame(data_list)
    print(f"📊 DataFrame 생성: {df.shape}")
    print(f"📋 컬럼 수: {len(df.columns)}")
    
    return df

# ============================================
# 실제 실행 코드
# ============================================

if __name__ == "__main__":
    # 1. 데이터 로드
    data_path = r"C:\WANTEDLAB_2025\AI AGENT08\petremote\dogwalk\2.라벨링데이터"
    df = load_pet_json_data(data_path)
    
    if df is not None:
        # 2. 기본 정보 확인
        print("\n📊 ===== 기본 데이터 정보 =====")
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()[:10]}...")  # 처음 10개 컬럼만
        
        # 3. 심각도 분포 확인
        if 'severity' in df.columns:
            print(f"\n🎯 Severity 분포:")
            print(df['severity'].value_counts().sort_index())
        
        # 4. 결측값 확인
        print(f"\n🔍 결측값 TOP 10:")
        missing_info = df.isnull().sum().sort_values(ascending=False)
        print(missing_info.head(10))
        
        # 5. 전처리 클래스 import (위에서 만든 코드 필요)
        # from pet_health_preprocessor import PetHealthPreprocessor
        
        # 6. 전처리 실행
        # preprocessor = PetHealthPreprocessor()
        # result = preprocessor.run_full_preprocessing(
        #     df=df, 
        #     problem_type='binary',
        #     imbalance_strategy='combined'
        # )
        
        # 7. 결과 저장
        # df.to_csv('pet_health_processed.csv', index=False)
        # print("✅ 처리된 데이터 저장 완료: pet_health_processed.csv")
        
        print("\n🎉 데이터 로딩 완료! 이제 전처리를 실행하세요.")
    
    else:
        print("❌ 데이터 로드 실패")

🔍 데이터 로딩 시작: C:\WANTEDLAB_2025\AI AGENT08\petremote\dogwalk\2.라벨링데이터
📁 발견된 JSON 파일: 1050개
📋 샘플 파일 구조 확인: SNC_2024_09_05_17_11_10_00014.json
🔍 샘플 데이터 구조:
   - 키 개수: 9
   - 주요 키들: ['image_info', 'annotation_info', 'pet_medical_record_info', 'sensor_values', 'timestamp', 'size', 'severity', 'age', 'dog_type']
   진행중... 1/1050
   진행중... 101/1050
   진행중... 201/1050
   진행중... 301/1050
   진행중... 401/1050
   진행중... 501/1050
   진행중... 601/1050
   진행중... 701/1050
   진행중... 801/1050
   진행중... 901/1050
   진행중... 1001/1050
✅ 총 1050개 파일 로드 완료
📊 DataFrame 생성: (1050, 10)
📋 컬럼 수: 10

📊 ===== 기본 데이터 정보 =====
Shape: (1050, 10)
Columns: ['image_info', 'annotation_info', 'pet_medical_record_info', 'sensor_values', 'timestamp', 'size', 'severity', 'age', 'dog_type', 'pet_id']...

🎯 Severity 분포:
severity
0    596
1     89
2    142
3    197
4     26
Name: count, dtype: int64

🔍 결측값 TOP 10:
image_info                 0
annotation_info            0
pet_medical_record_info    0
sensor_values              0
times

In [10]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

class RealPetDataPreprocessor:
    def __init__(self):
        self.scaler = RobustScaler()
        self.imputer = KNNImputer(n_neighbors=5)
        self.label_encoder = LabelEncoder()
        self.feature_names = None
        
    def load_and_analyze_structure(self, df):
        """
        1단계: 실제 데이터 구조 분석
        """
        print("🔍 ===== 실제 데이터 구조 분석 =====")
        print(f"📊 DataFrame 형태: {df.shape}")
        print(f"📋 컬럼들: {list(df.columns)}")
        
        # 각 컬럼의 데이터 타입과 샘플 확인
        for col in df.columns:
            print(f"\n📌 {col}:")
            print(f"   - 타입: {type(df[col].iloc[0])}")
            
            # JSON/dict 형태인 경우 내부 구조 확인
            if isinstance(df[col].iloc[0], (dict, list)):
                if isinstance(df[col].iloc[0], dict):
                    sample_keys = list(df[col].iloc[0].keys())[:5]
                    print(f"   - 키들: {sample_keys}...")
                elif isinstance(df[col].iloc[0], list):
                    print(f"   - 리스트 길이: {len(df[col].iloc[0])}")
            else:
                print(f"   - 샘플 값: {df[col].iloc[0]}")
        
        return df
    
    def extract_pose_features(self, df):
        """
        2단계: annotation_info에서 포즈 특성 추출
        """
        print("\n🦴 ===== 포즈 특성 추출 =====")
        
        pose_features = []
        
        for idx, row in df.iterrows():
            if idx % 200 == 0:
                print(f"   진행중... {idx+1}/{len(df)}")
            
            pose_dict = {'pet_id': row['pet_id']}
            
            try:
                annotation_info = row['annotation_info']
                
                # annotation_info가 dict인 경우 포즈 포인트 추출
                if isinstance(annotation_info, dict):
                    # 키를 확인해서 포즈 포인트들 찾기
                    for key, value in annotation_info.items():
                        if isinstance(value, dict):
                            # P0, P1 등의 포즈 포인트라고 가정
                            if 'x' in value and 'y' in value:
                                pose_dict[f'{key}_x'] = value.get('x', np.nan)
                                pose_dict[f'{key}_y'] = value.get('y', np.nan)
                                pose_dict[f'{key}_conf'] = value.get('confidence', value.get('conf', 1.0))
                
                # 또는 리스트 형태인 경우
                elif isinstance(annotation_info, list):
                    for i, point in enumerate(annotation_info):
                        if isinstance(point, dict) and 'x' in point and 'y' in point:
                            pose_dict[f'P{i}_x'] = point.get('x', np.nan)
                            pose_dict[f'P{i}_y'] = point.get('y', np.nan)
                            pose_dict[f'P{i}_conf'] = point.get('confidence', point.get('conf', 1.0))
                            
            except Exception as e:
                print(f"   ⚠️  {idx}번째 행 포즈 추출 실패: {e}")
                # 기본값으로 채우기
                for i in range(13):  # P0~P12
                    pose_dict[f'P{i}_x'] = np.nan
                    pose_dict[f'P{i}_y'] = np.nan
                    pose_dict[f'P{i}_conf'] = np.nan
            
            pose_features.append(pose_dict)
        
        pose_df = pd.DataFrame(pose_features)
        print(f"✅ 포즈 특성 추출 완료: {pose_df.shape}")
        
        # 결측값 확인
        missing_info = pose_df.isnull().sum().sort_values(ascending=False)
        high_missing = missing_info[missing_info > len(pose_df) * 0.5]
        if len(high_missing) > 0:
            print(f"📊 50% 이상 결측인 포즈 특성: {len(high_missing)}개")
            print(high_missing.head())
        
        return pose_df
    
    def extract_sensor_features(self, df):
        """
        3단계: sensor_values에서 센서 특성 추출
        """
        print("\n📱 ===== 센서 특성 추출 =====")
        
        sensor_features = []
        
        for idx, row in df.iterrows():
            if idx % 200 == 0:
                print(f"   진행중... {idx+1}/{len(df)}")
                
            sensor_dict = {'pet_id': row['pet_id']}
            
            try:
                sensor_values = row['sensor_values']
                
                if isinstance(sensor_values, dict):
                    # 각 센서별로 통계적 특성 계산
                    for sensor_name, values in sensor_values.items():
                        if isinstance(values, list) and len(values) > 0:
                            values_array = np.array(values)
                            
                            # 기본 통계 특성
                            sensor_dict[f'{sensor_name}_mean'] = np.mean(values_array)
                            sensor_dict[f'{sensor_name}_std'] = np.std(values_array)
                            sensor_dict[f'{sensor_name}_min'] = np.min(values_array)
                            sensor_dict[f'{sensor_name}_max'] = np.max(values_array)
                            sensor_dict[f'{sensor_name}_median'] = np.median(values_array)
                            
                            # 의료 특화 특성
                            sensor_dict[f'{sensor_name}_range'] = np.max(values_array) - np.min(values_array)
                            sensor_dict[f'{sensor_name}_cv'] = np.std(values_array) / (np.mean(values_array) + 1e-6)  # 변동계수
                            
                            # 분포 특성
                            sensor_dict[f'{sensor_name}_skew'] = pd.Series(values_array).skew()
                            sensor_dict[f'{sensor_name}_kurtosis'] = pd.Series(values_array).kurtosis()
                            
                            # 주파수 도메인 특성 (FFT)
                            fft_values = np.fft.fft(values_array)
                            fft_magnitude = np.abs(fft_values)
                            sensor_dict[f'{sensor_name}_dominant_freq'] = np.argmax(fft_magnitude[1:len(fft_magnitude)//2]) + 1
                            sensor_dict[f'{sensor_name}_spectral_energy'] = np.sum(fft_magnitude**2)
                        
                        else:
                            # 단일 값인 경우
                            sensor_dict[f'{sensor_name}_value'] = values
                            
            except Exception as e:
                print(f"   ⚠️  {idx}번째 행 센서 추출 실패: {e}")
                # 기본값으로 채우기
                for sensor in ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']:
                    for stat in ['mean', 'std', 'min', 'max', 'median', 'range', 'cv', 'skew', 'kurtosis']:
                        sensor_dict[f'{sensor}_{stat}'] = np.nan
            
            sensor_features.append(sensor_dict)
        
        sensor_df = pd.DataFrame(sensor_features)
        print(f"✅ 센서 특성 추출 완료: {sensor_df.shape}")
        
        return sensor_df
    
    def extract_medical_features(self, df):
        """
        4단계: pet_medical_record_info에서 의료 특성 추출
        """
        print("\n🏥 ===== 의료 기록 특성 추출 =====")
        
        medical_features = []
        
        for idx, row in df.iterrows():
            if idx % 200 == 0:
                print(f"   진행중... {idx+1}/{len(df)}")
                
            medical_dict = {'pet_id': row['pet_id']}
            
            try:
                medical_info = row['pet_medical_record_info']
                
                if isinstance(medical_info, dict):
                    # 의료 기록에서 특성 추출
                    for key, value in medical_info.items():
                        if isinstance(value, (int, float)):
                            medical_dict[f'medical_{key}'] = value
                        elif isinstance(value, str):
                            # 문자열은 라벨 인코딩 예정
                            medical_dict[f'medical_{key}'] = value
                        elif isinstance(value, list):
                            # 리스트는 통계 특성으로 변환
                            if len(value) > 0 and all(isinstance(v, (int, float)) for v in value):
                                medical_dict[f'medical_{key}_mean'] = np.mean(value)
                                medical_dict[f'medical_{key}_std'] = np.std(value)
                                
            except Exception as e:
                print(f"   ⚠️  {idx}번째 행 의료 기록 추출 실패: {e}")
            
            medical_features.append(medical_dict)
        
        medical_df = pd.DataFrame(medical_features)
        print(f"✅ 의료 기록 특성 추출 완료: {medical_df.shape}")
        
        return medical_df
    
    def create_basic_features(self, df):
        """
        5단계: 기본 특성 정리
        """
        print("\n📋 ===== 기본 특성 정리 =====")
        
        basic_df = df[['pet_id', 'severity', 'age', 'dog_type', 'size']].copy()
        
        # 타겟 변수 확인
        print(f"🎯 Severity 분포:")
        severity_dist = basic_df['severity'].value_counts().sort_index()
        for sev, count in severity_dist.items():
            print(f"   Level {sev}: {count:3d}개 ({count/len(basic_df)*100:5.1f}%)")
        
        # 품종별 분포 확인
        print(f"\n🐕 품종별 분포 (TOP 10):")
        breed_dist = basic_df['dog_type'].value_counts().head(10)
        for breed, count in breed_dist.items():
            print(f"   {breed}: {count}개")
        
        return basic_df
    
    def feature_engineering(self, merged_df):
        """
        6단계: 의료 AI 특화 특성 엔지니어링
        """
        print("\n⚙️ ===== 의료 특화 특성 엔지니어링 =====")
        
        # 센서 기반 의료 지표
        sensor_cols = [col for col in merged_df.columns if any(sensor in col for sensor in ['acc', 'gyro'])]
        
        if sensor_cols:
            # 움직임 안정성 지표
            std_cols = [col for col in sensor_cols if 'std' in col]
            if std_cols:
                merged_df['movement_instability'] = merged_df[std_cols].mean(axis=1)
                merged_df['movement_stability'] = 1 / (merged_df['movement_instability'] + 1e-6)
                print("   ✅ 움직임 안정성 지표 생성")
            
            # 활동 강도 지표
            mean_cols = [col for col in sensor_cols if 'mean' in col and 'acc' in col]
            if len(mean_cols) >= 3:
                merged_df['activity_intensity'] = np.sqrt(
                    merged_df[mean_cols[0]]**2 + 
                    merged_df[mean_cols[1]]**2 + 
                    merged_df[mean_cols[2]]**2
                )
                print("   ✅ 활동 강도 지표 생성")
            
            # 센서 불규칙성 지표
            cv_cols = [col for col in sensor_cols if 'cv' in col]
            if cv_cols:
                merged_df['sensor_irregularity'] = merged_df[cv_cols].mean(axis=1)
                print("   ✅ 센서 불규칙성 지표 생성")
        
        # 포즈 기반 의료 지표
        conf_cols = [col for col in merged_df.columns if 'conf' in col]
        if conf_cols:
            merged_df['pose_confidence_mean'] = merged_df[conf_cols].mean(axis=1)
            merged_df['pose_confidence_std'] = merged_df[conf_cols].std(axis=1)
            merged_df['pose_detection_rate'] = merged_df[conf_cols].notna().sum(axis=1) / len(conf_cols)
            print("   ✅ 포즈 신뢰도 지표 생성")
        
        # 포즈 대칭성 지표 (좌우 관절 비교)
        # 가정: P1,P2,P3는 왼쪽, P4,P5,P6는 오른쪽 다리
        left_joints = [col for col in merged_df.columns if any(f'P{i}_' in col for i in [1,2,3])]
        right_joints = [col for col in merged_df.columns if any(f'P{i}_' in col for i in [4,5,6])]
        
        if left_joints and right_joints:
            # 좌우 대칭성 계산 (간단한 버전)
            left_mean = merged_df[left_joints].mean(axis=1)
            right_mean = merged_df[right_joints].mean(axis=1)
            merged_df['pose_asymmetry'] = np.abs(left_mean - right_mean)
            print("   ✅ 포즈 대칭성 지표 생성")
        
        # 품종별 정규화
        if 'dog_type' in merged_df.columns:
            for feature in ['age', 'size']:
                if feature in merged_df.columns:
                    merged_df[f'{feature}_percentile_by_breed'] = merged_df.groupby('dog_type')[feature].rank(pct=True)
                    print(f"   ✅ {feature} 품종별 백분위수 생성")
        
        print(f"✅ 특성 엔지니어링 완료. 최종 특성 수: {len(merged_df.columns)}")
        return merged_df
    
    def handle_missing_values(self, df):
        """
        7단계: 결측값 처리
        """
        print("\n🔧 ===== 결측값 처리 =====")
        
        # 결측값 분석
        missing_info = df.isnull().sum().sort_values(ascending=False)
        high_missing = missing_info[missing_info > len(df) * 0.8]
        
        if len(high_missing) > 0:
            print(f"📊 80% 이상 결측인 특성들 ({len(high_missing)}개):")
            for col, missing_count in high_missing.items():
                missing_pct = missing_count / len(df) * 100
                print(f"   {col}: {missing_pct:.1f}%")
            
            # 80% 이상 결측인 특성들은 제거하거나 플래그로 변환
            for col in high_missing.index:
                if 'P12' in col:  # P12는 특별히 플래그로 변환
                    df[f'{col}_available'] = (~df[col].isna()).astype(int)
                df = df.drop(columns=[col])
            
            print(f"   ✅ 고결측 특성 {len(high_missing)}개 처리 완료")
        
        # 나머지 결측값들 처리
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        missing_cols = [col for col in numeric_cols if df[col].isnull().sum() > 0]
        
        if missing_cols:
            print(f"📌 {len(missing_cols)}개 특성의 결측값을 KNN으로 보간")
            df[missing_cols] = self.imputer.fit_transform(df[missing_cols])
            print("   ✅ KNN 보간 완료")
        
        print(f"✅ 최종 결측값 개수: {df.isnull().sum().sum()}")
        return df
    
    def prepare_for_modeling(self, df):
        """
        8단계: 모델링을 위한 최종 준비
        """
        print("\n🎯 ===== 모델링 준비 =====")
        
        # 타겟 변수 분리
        if 'severity' in df.columns:
            y = df['severity'].copy()
            X = df.drop(columns=['severity', 'pet_id'])
        else:
            raise ValueError("severity 컬럼을 찾을 수 없습니다.")
        
        # 범주형 변수 인코딩
        categorical_cols = X.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            X[col] = self.label_encoder.fit_transform(X[col].astype(str))
            print(f"   ✅ {col} 라벨 인코딩 완료")
        
        # 특성명 저장
        self.feature_names = X.columns.tolist()
        
        print(f"✅ 최종 특성 수: {len(self.feature_names)}")
        print(f"✅ 데이터 형태: X{X.shape}, y{y.shape}")
        
        return X, y
    
    def run_full_preprocessing(self, df):
        """
        전체 전처리 파이프라인 실행
        """
        print("🚀 ===== 실제 반려동물 데이터 전처리 시작 =====")
        
        # 1단계: 데이터 구조 분석
        df = self.load_and_analyze_structure(df)
        
        # 2단계: 각 특성별 추출
        basic_df = self.create_basic_features(df)
        pose_df = self.extract_pose_features(df)
        sensor_df = self.extract_sensor_features(df)
        medical_df = self.extract_medical_features(df)
        
        # 3단계: 모든 특성 병합
        print("\n🔗 ===== 특성 병합 =====")
        merged_df = basic_df.copy()
        
        # pet_id 기준으로 병합
        for feature_df in [pose_df, sensor_df, medical_df]:
            merged_df = merged_df.merge(feature_df, on='pet_id', how='left')
            print(f"   병합 후 형태: {merged_df.shape}")
        
        # 4단계: 특성 엔지니어링
        merged_df = self.feature_engineering(merged_df)
        
        # 5단계: 결측값 처리
        merged_df = self.handle_missing_values(merged_df)
        
        # 6단계: 모델링 준비
        X, y = self.prepare_for_modeling(merged_df)
        
        # 7단계: 데이터 분할
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # 8단계: 스케일링
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
        X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
        
        print(f"\n🎉 ===== 전처리 완료 =====")
        print(f"✅ 훈련 데이터: {X_train_scaled.shape}")
        print(f"✅ 테스트 데이터: {X_test_scaled.shape}")
        print(f"✅ 특성 수: {len(self.feature_names)}")
        print(f"✅ 클래스 분포: {pd.Series(y_train).value_counts().sort_index().to_dict()}")
        
        return {
            'X_train': X_train_scaled,
            'X_test': X_test_scaled,
            'y_train': y_train,
            'y_test': y_test,
            'feature_names': self.feature_names,
            'scaler': self.scaler,
            'raw_data': merged_df
        }

# ===============================================
# 사용 예시
# ===============================================

def run_real_preprocessing(df):
    """
    실제 데이터 전처리 실행
    """
    preprocessor = RealPetDataPreprocessor()
    result = preprocessor.run_full_preprocessing(df)
    
    # 결과 저장
    result['raw_data'].to_csv('processed_pet_data.csv', index=False)
    print("✅ 전처리된 데이터 저장: processed_pet_data.csv")
    
    return result

# 실행 방법:
# result = run_real_preprocessing(df)
# X_train = result['X_train']
# y_train = result['y_train']

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

class FixedPetDataPreprocessor:
    def __init__(self):
        self.scaler = RobustScaler()
        self.imputer = KNNImputer(n_neighbors=5)
        self.label_encoders = {}
        
    def extract_pose_features_fixed(self, df):
        """수정된 포즈 특성 추출"""
        print("🦴 ===== 수정된 포즈 특성 추출 =====")
        
        pose_features = []
        
        for idx, row in df.iterrows():
            if idx % 200 == 0:
                print(f"   진행중... {idx+1}/{len(df)}")
            
            pose_dict = {'pet_id': row['pet_id']}
            
            try:
                annotation_info = row['annotation_info']
                
                # 리스트 형태의 annotation_info 처리
                if isinstance(annotation_info, list):
                    for i, point in enumerate(annotation_info):
                        if isinstance(point, dict):
                            # x, y 좌표를 float로 변환
                            x_val = float(point.get('x', 0)) if point.get('x') != '' else np.nan
                            y_val = float(point.get('y', 0)) if point.get('y') != '' else np.nan
                            
                            pose_dict[f'P{i}_x'] = x_val
                            pose_dict[f'P{i}_y'] = y_val
                            pose_dict[f'P{i}_label'] = point.get('label', f'Point_{i}')
                            # 신뢰도는 좌표가 있으면 1, 없으면 0
                            pose_dict[f'P{i}_conf'] = 1.0 if not (np.isnan(x_val) or np.isnan(y_val)) else 0.0
                            
            except Exception as e:
                print(f"   ⚠️  {idx}번째 행 포즈 추출 실패: {e}")
                # 기본값으로 채우기
                for i in range(7):  # 리스트 길이가 7이라고 했으니
                    pose_dict[f'P{i}_x'] = np.nan
                    pose_dict[f'P{i}_y'] = np.nan
                    pose_dict[f'P{i}_label'] = f'Point_{i}'
                    pose_dict[f'P{i}_conf'] = 0.0
            
            pose_features.append(pose_dict)
        
        pose_df = pd.DataFrame(pose_features)
        print(f"✅ 포즈 특성 추출 완료: {pose_df.shape}")
        return pose_df
    
    def extract_sensor_features_fixed(self, df):
        """수정된 센서 특성 추출"""
        print("📱 ===== 수정된 센서 특성 추출 =====")
        
        sensor_features = []
        
        for idx, row in df.iterrows():
            if idx % 200 == 0:
                print(f"   진행중... {idx+1}/{len(df)}")
                
            sensor_dict = {'pet_id': row['pet_id']}
            
            try:
                sensor_values = row['sensor_values']
                
                # 리스트 형태의 센서 데이터 처리
                if isinstance(sensor_values, list):
                    # 전체 센서 데이터를 하나의 시계열로 보고 통계 계산
                    all_values = []
                    for sensor_array in sensor_values:
                        if isinstance(sensor_array, list):
                            all_values.extend(sensor_array)
                    
                    if all_values:
                        values_array = np.array(all_values, dtype=float)
                        
                        # 기본 통계 특성
                        sensor_dict['sensor_mean'] = np.mean(values_array)
                        sensor_dict['sensor_std'] = np.std(values_array)
                        sensor_dict['sensor_min'] = np.min(values_array)
                        sensor_dict['sensor_max'] = np.max(values_array)
                        sensor_dict['sensor_median'] = np.median(values_array)
                        sensor_dict['sensor_range'] = np.max(values_array) - np.min(values_array)
                        sensor_dict['sensor_var'] = np.var(values_array)
                        
                        # 각 센서 배열별 통계 (첫 3개만)
                        for i, sensor_array in enumerate(sensor_values[:3]):
                            if isinstance(sensor_array, list):
                                arr = np.array(sensor_array, dtype=float)
                                sensor_dict[f'sensor_{i}_mean'] = np.mean(arr)
                                sensor_dict[f'sensor_{i}_std'] = np.std(arr)
                                sensor_dict[f'sensor_{i}_max'] = np.max(arr)
                    
                    else:
                        # 빈 데이터인 경우
                        for key in ['sensor_mean', 'sensor_std', 'sensor_min', 'sensor_max', 
                                   'sensor_median', 'sensor_range', 'sensor_var']:
                            sensor_dict[key] = np.nan
                            
            except Exception as e:
                print(f"   ⚠️  {idx}번째 행 센서 추출 실패: {e}")
                # 기본값으로 채우기
                for key in ['sensor_mean', 'sensor_std', 'sensor_min', 'sensor_max', 
                           'sensor_median', 'sensor_range', 'sensor_var']:
                    sensor_dict[key] = np.nan
            
            sensor_features.append(sensor_dict)
        
        sensor_df = pd.DataFrame(sensor_features)
        print(f"✅ 센서 특성 추출 완료: {sensor_df.shape}")
        return sensor_df
    
    def extract_medical_features_fixed(self, df):
        """수정된 의료 기록 특성 추출"""
        print("🏥 ===== 수정된 의료 기록 특성 추출 =====")
        
        medical_features = []
        
        for idx, row in df.iterrows():
            if idx % 200 == 0:
                print(f"   진행중... {idx+1}/{len(df)}")
                
            medical_dict = {'pet_id': row['pet_id']}
            
            try:
                medical_info = row['pet_medical_record_info']
                
                # 리스트 형태의 의료 기록 처리
                if isinstance(medical_info, list):
                    for i, record in enumerate(medical_info):
                        if isinstance(record, dict):
                            for key, value in record.items():
                                try:
                                    # 숫자로 변환 가능한 경우
                                    medical_dict[f'medical_{key}_{i}'] = float(value)
                                except (ValueError, TypeError):
                                    # 문자열인 경우 나중에 라벨 인코딩
                                    medical_dict[f'medical_{key}_{i}'] = str(value)
                                    
            except Exception as e:
                print(f"   ⚠️  {idx}번째 행 의료 기록 추출 실패: {e}")
            
            medical_features.append(medical_dict)
        
        medical_df = pd.DataFrame(medical_features)
        print(f"✅ 의료 기록 특성 추출 완료: {medical_df.shape}")
        return medical_df
    
    def create_basic_features_fixed(self, df):
        """수정된 기본 특성 정리"""
        print("📋 ===== 수정된 기본 특성 정리 =====")
        
        basic_df = df[['pet_id', 'severity', 'age', 'dog_type', 'size', 'timestamp']].copy()
        
        # age를 숫자로 변환
        age_mapping = {
            '유령견': 1, '유년견': 1, '어린견': 1,  # 어린 개
            '성견': 2, '성년견': 2,  # 성년 개  
            '노령견': 3, '고령견': 3, '노년견': 3  # 노년 개
        }
        
        basic_df['age_numeric'] = basic_df['age'].map(age_mapping)
        # 매핑되지 않은 경우 기본값
        basic_df['age_numeric'] = basic_df['age_numeric'].fillna(2)
        
        # size를 숫자로 변환
        size_mapping = {
            '소형견': 1, '초소형견': 0,
            '중형견': 2, '중소형견': 1.5,
            '대형견': 3, '초대형견': 4
        }
        
        basic_df['size_numeric'] = basic_df['size'].map(size_mapping)
        basic_df['size_numeric'] = basic_df['size_numeric'].fillna(2)  # 기본값은 중형견
        
        # timestamp를 정규화 (선택사항)
        basic_df['timestamp_normalized'] = (basic_df['timestamp'] - basic_df['timestamp'].min()) / (basic_df['timestamp'].max() - basic_df['timestamp'].min())
        
        print(f"🎯 Severity 분포:")
        severity_dist = basic_df['severity'].value_counts().sort_index()
        for sev, count in severity_dist.items():
            print(f"   Level {sev}: {count:3d}개 ({count/len(basic_df)*100:5.1f}%)")
        
        print(f"📊 Age 매핑 결과:")
        age_dist = basic_df['age_numeric'].value_counts().sort_index()
        for age_num, count in age_dist.items():
            print(f"   Age {age_num}: {count}개")
        
        return basic_df
    
    def feature_engineering_fixed(self, merged_df):
        """수정된 특성 엔지니어링"""
        print("⚙️ ===== 수정된 특성 엔지니어링 =====")
        
        # 포즈 신뢰도 지표
        conf_cols = [col for col in merged_df.columns if 'conf' in col]
        if conf_cols:
            merged_df['pose_confidence_mean'] = merged_df[conf_cols].mean(axis=1)
            merged_df['pose_confidence_std'] = merged_df[conf_cols].std(axis=1)
            merged_df['pose_detection_rate'] = merged_df[conf_cols].mean(axis=1)
            print("   ✅ 포즈 신뢰도 지표 생성")
        
        # 센서 기반 지표
        sensor_cols = [col for col in merged_df.columns if 'sensor_' in col and col.endswith('_std')]
        if sensor_cols:
            merged_df['sensor_instability'] = merged_df[sensor_cols].mean(axis=1)
            merged_df['sensor_stability'] = 1 / (merged_df['sensor_instability'] + 1e-6)
            print("   ✅ 센서 안정성 지표 생성")
        
        # 포즈 좌표 범위 계산
        x_cols = [col for col in merged_df.columns if col.endswith('_x')]
        y_cols = [col for col in merged_df.columns if col.endswith('_y')]
        
        if x_cols and y_cols:
            merged_df['pose_x_range'] = merged_df[x_cols].max(axis=1) - merged_df[x_cols].min(axis=1)
            merged_df['pose_y_range'] = merged_df[y_cols].max(axis=1) - merged_df[y_cols].min(axis=1)
            merged_df['pose_area'] = merged_df['pose_x_range'] * merged_df['pose_y_range']
            print("   ✅ 포즈 공간 지표 생성")
        
        # 품종별 정규화
        if 'dog_type' in merged_df.columns:
            for feature in ['age_numeric', 'size_numeric']:
                if feature in merged_df.columns:
                    merged_df[f'{feature}_breed_norm'] = merged_df.groupby('dog_type')[feature].transform(
                        lambda x: (x - x.mean()) / (x.std() + 1e-6)
                    )
                    print(f"   ✅ {feature} 품종별 정규화 생성")
        
        print(f"✅ 특성 엔지니어링 완료. 최종 특성 수: {len(merged_df.columns)}")
        return merged_df
    
    def run_fixed_preprocessing(self, df):
        """수정된 전체 전처리 파이프라인"""
        print("🚀 ===== 수정된 전처리 시작 =====")
        
        try:
            # 1. 기본 특성 처리
            basic_df = self.create_basic_features_fixed(df)
            
            # 2. 포즈 특성 추출
            pose_df = self.extract_pose_features_fixed(df)
            
            # 3. 센서 특성 추출  
            sensor_df = self.extract_sensor_features_fixed(df)
            
            # 4. 의료 기록 특성 추출
            medical_df = self.extract_medical_features_fixed(df)
            
            # 5. 모든 특성 병합
            print("\n🔗 ===== 특성 병합 =====")
            merged_df = basic_df.copy()
            
            for feature_df in [pose_df, sensor_df, medical_df]:
                merged_df = merged_df.merge(feature_df, on='pet_id', how='left')
                print(f"   병합 후 형태: {merged_df.shape}")
            
            # 6. 특성 엔지니어링
            merged_df = self.feature_engineering_fixed(merged_df)
            
            # 7. 범주형 변수 인코딩
            categorical_cols = merged_df.select_dtypes(include=['object']).columns
            categorical_cols = [col for col in categorical_cols if col not in ['pet_id']]
            
            for col in categorical_cols:
                if col not in self.label_encoders:
                    self.label_encoders[col] = LabelEncoder()
                merged_df[col] = self.label_encoders[col].fit_transform(merged_df[col].astype(str))
                print(f"   ✅ {col} 라벨 인코딩 완료")
            
            # 8. 타겟과 특성 분리
            y = merged_df['severity'].copy()
            X = merged_df.drop(columns=['severity', 'pet_id'])
            
            # 9. 결측값 처리
            numeric_cols = X.select_dtypes(include=[np.number]).columns
            X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
            
            # 10. 데이터 분할
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            
            # 11. 스케일링
            X_train_scaled = self.scaler.fit_transform(X_train)
            X_test_scaled = self.scaler.transform(X_test)
            
            X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
            X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
            
            print(f"\n🎉 ===== 수정된 전처리 완료 =====")
            print(f"✅ 훈련 데이터: {X_train_scaled.shape}")
            print(f"✅ 테스트 데이터: {X_test_scaled.shape}")
            print(f"✅ 특성 수: {len(X_train.columns)}")
            print(f"✅ 클래스 분포: {pd.Series(y_train).value_counts().sort_index().to_dict()}")
            
            return {
                'X_train': X_train_scaled,
                'X_test': X_test_scaled,
                'y_train': y_train,
                'y_test': y_test,
                'feature_names': list(X_train.columns),
                'scaler': self.scaler,
                'raw_data': merged_df
            }
            
        except Exception as e:
            print(f"❌ 수정된 전처리 중 오류 발생: {e}")
            import traceback
            traceback.print_exc()
            return None

# ===== 실행 코드 =====
def run_fixed_preprocessing():
    print("🔧 수정된 전처리 실행...")
    
    fixed_preprocessor = FixedPetDataPreprocessor()
    result = fixed_preprocessor.run_fixed_preprocessing(df)
    
    if result is not None:
        print("\n✅ 전처리 성공!")
        
        # 결과 저장
        result['raw_data'].to_csv('fixed_pet_data_1050.csv', index=False)
        print("💾 결과 저장: fixed_pet_data_1050.csv")
        
        return result
    else:
        print("❌ 전처리 실패")
        return None

# 실행!
result = run_fixed_preprocessing()

🔧 수정된 전처리 실행...
🚀 ===== 수정된 전처리 시작 =====
📋 ===== 수정된 기본 특성 정리 =====
🎯 Severity 분포:
   Level 0: 596개 ( 56.8%)
   Level 1:  89개 (  8.5%)
   Level 2: 142개 ( 13.5%)
   Level 3: 197개 ( 18.8%)
   Level 4:  26개 (  2.5%)
📊 Age 매핑 결과:
   Age 2.0: 430개
   Age 3.0: 620개
🦴 ===== 수정된 포즈 특성 추출 =====
   진행중... 1/1050
   진행중... 201/1050
   진행중... 401/1050
   진행중... 601/1050
   진행중... 801/1050
   진행중... 1001/1050
✅ 포즈 특성 추출 완료: (1050, 53)
📱 ===== 수정된 센서 특성 추출 =====
   진행중... 1/1050
   진행중... 201/1050
   진행중... 401/1050
   진행중... 601/1050
   진행중... 801/1050
   진행중... 1001/1050
✅ 센서 특성 추출 완료: (1050, 17)
🏥 ===== 수정된 의료 기록 특성 추출 =====
   진행중... 1/1050
   진행중... 201/1050
   진행중... 401/1050
   진행중... 601/1050
   진행중... 801/1050
   진행중... 1001/1050
✅ 의료 기록 특성 추출 완료: (1050, 5)

🔗 ===== 특성 병합 =====
   병합 후 형태: (1050, 61)
   병합 후 형태: (1050, 77)
   병합 후 형태: (1050, 81)
⚙️ ===== 수정된 특성 엔지니어링 =====
   ✅ 포즈 신뢰도 지표 생성
   ✅ 센서 안정성 지표 생성
   ✅ 포즈 공간 지표 생성
   ✅ age_numeric 품종별 정규화 생성
   ✅ size_numeric 품종별 정규화 생성
✅ 특성 엔지니어

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

def analyze_preprocessed_data(csv_path='fixed_pet_data_1050.csv'):
    """
    전처리된 데이터 상세 분석
    """
    print("📊 ===== 전처리 결과 분석 시작 =====")
    
    # 1. 데이터 로드
    try:
        df = pd.read_csv(csv_path)
        print(f"✅ 데이터 로드 성공: {df.shape}")
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        return None
    except Exception as e:
        print(f"❌ 데이터 로드 실패: {e}")
        return None
    
    # 2. 기본 정보
    print(f"\n📋 ===== 기본 정보 =====")
    print(f"📊 데이터 형태: {df.shape}")
    print(f"📈 총 특성 수: {len(df.columns)}")
    print(f"🎯 타겟 변수: {'severity' if 'severity' in df.columns else '없음'}")
    print(f"🔍 메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # 3. 컬럼 분류
    print(f"\n📂 ===== 특성 분류 =====")
    
    # 특성 유형별 분류
    basic_cols = [col for col in df.columns if col in ['pet_id', 'severity', 'age', 'dog_type', 'size', 'timestamp']]
    pose_cols = [col for col in df.columns if any(pattern in col for pattern in ['P0_', 'P1_', 'P2_', 'P3_', 'P4_', 'P5_', 'P6_'])]
    sensor_cols = [col for col in df.columns if 'sensor' in col]
    medical_cols = [col for col in df.columns if 'medical' in col]
    engineered_cols = [col for col in df.columns if any(pattern in col for pattern in ['confidence', 'stability', 'range', 'area', 'norm'])]
    
    print(f"📋 기본 정보 컬럼: {len(basic_cols)}개")
    print(f"   {basic_cols[:5]}{'...' if len(basic_cols) > 5 else ''}")
    
    print(f"🦴 포즈 관련 컬럼: {len(pose_cols)}개")
    print(f"   {pose_cols[:5]}{'...' if len(pose_cols) > 5 else ''}")
    
    print(f"📱 센서 관련 컬럼: {len(sensor_cols)}개")
    print(f"   {sensor_cols[:5]}{'...' if len(sensor_cols) > 5 else ''}")
    
    print(f"🏥 의료 기록 컬럼: {len(medical_cols)}개")
    print(f"   {medical_cols[:5]}{'...' if len(medical_cols) > 5 else ''}")
    
    print(f"⚙️ 엔지니어링 컬럼: {len(engineered_cols)}개")
    print(f"   {engineered_cols[:5]}{'...' if len(engineered_cols) > 5 else ''}")
    
    # 4. 타겟 변수 분석
    if 'severity' in df.columns:
        print(f"\n🎯 ===== 타겟 변수 분석 =====")
        severity_dist = df['severity'].value_counts().sort_index()
        total = len(df)
        
        print(f"📊 심각도 분포:")
        for level, count in severity_dist.items():
            percentage = count / total * 100
            bar = "█" * int(percentage / 2)
            print(f"   Level {level}: {count:3d}개 ({percentage:5.1f}%) {bar}")
        
        # 클래스 불균형 비율
        max_class = severity_dist.max()
        min_class = severity_dist.min()
        imbalance_ratio = max_class / min_class
        print(f"📈 클래스 불균형 비율: {imbalance_ratio:.1f}:1")
        
        # 이진분류로 변환시 분포
        binary_normal = (df['severity'] == 0).sum()
        binary_disease = (df['severity'] > 0).sum()
        print(f"🔄 이진분류 분포: 정상 {binary_normal}개 vs 질병 {binary_disease}개 ({binary_disease/(binary_normal+binary_disease)*100:.1f}%)")
    
    # 5. 결측값 분석
    print(f"\n🔍 ===== 결측값 분석 =====")
    missing_info = df.isnull().sum()
    missing_cols = missing_info[missing_info > 0].sort_values(ascending=False)
    
    if len(missing_cols) > 0:
        print(f"❌ 결측값이 있는 컬럼: {len(missing_cols)}개")
        for col, missing_count in missing_cols.head(10).items():
            missing_pct = missing_count / len(df) * 100
            print(f"   {col}: {missing_count}개 ({missing_pct:.1f}%)")
    else:
        print(f"✅ 결측값 없음! 모든 컬럼 완전 처리됨")
    
    # 6. 데이터 타입 분석
    print(f"\n📊 ===== 데이터 타입 분석 =====")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   {dtype}: {count}개 컬럼")
    
    # 7. 기본 통계
    print(f"\n📈 ===== 주요 특성 통계 =====")
    
    # 수치형 컬럼만 선택
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'severity']
    
    if len(numeric_cols) > 0:
        stats_df = df[numeric_cols].describe()
        
        # 분산이 큰 특성들 (중요할 가능성)
        high_var_features = df[numeric_cols].var().nlargest(10)
        print(f"📊 분산이 큰 특성 TOP 10:")
        for feature, variance in high_var_features.items():
            print(f"   {feature}: {variance:.2f}")
        
        # 범위가 큰 특성들
        print(f"\n📏 범위가 큰 특성 TOP 10:")
        ranges = df[numeric_cols].max() - df[numeric_cols].min()
        large_ranges = ranges.nlargest(10)
        for feature, range_val in large_ranges.items():
            print(f"   {feature}: {range_val:.2f}")
    
    # 8. 특성 간 상관관계 (severity와)
    if 'severity' in df.columns and len(numeric_cols) > 0:
        print(f"\n🔗 ===== Severity와 상관관계 TOP 15 =====")
        
        correlations = df[numeric_cols + ['severity']].corr()['severity'].abs().sort_values(ascending=False)
        correlations = correlations.drop('severity')  # 자기 자신 제거
        
        print(f"📊 양의 상관관계 (높을수록 심각):")
        positive_corr = df[numeric_cols + ['severity']].corr()['severity'].sort_values(ascending=False)
        positive_corr = positive_corr.drop('severity')
        for feature, corr in positive_corr.head(8).items():
            bar = "█" * int(abs(corr) * 20)
            print(f"   {feature[:30]:30s}: {corr:+.3f} {bar}")
        
        print(f"\n📊 음의 상관관계 (낮을수록 심각):")
        negative_corr = positive_corr.tail(7)
        for feature, corr in negative_corr.items():
            bar = "█" * int(abs(corr) * 20)
            print(f"   {feature[:30]:30s}: {corr:+.3f} {bar}")
    
    # 9. 이상값 분석
    print(f"\n⚠️ ===== 이상값 분석 =====")
    if len(numeric_cols) > 0:
        outlier_counts = {}
        
        for col in numeric_cols[:20]:  # 처음 20개만 확인
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            outlier_counts[col] = len(outliers)
        
        # 이상값이 많은 특성들
        high_outliers = sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)[:10]
        print(f"📊 이상값이 많은 특성 TOP 10:")
        for feature, count in high_outliers:
            percentage = count / len(df) * 100
            print(f"   {feature[:30]:30s}: {count:3d}개 ({percentage:4.1f}%)")
    
    # 10. 품종별 분석 (dog_type이 있는 경우)
    if 'dog_type' in df.columns:
        print(f"\n🐕 ===== 품종별 분석 =====")
        breed_counts = df['dog_type'].value_counts()
        print(f"📊 품종 개수: {len(breed_counts)}개")
        print(f"📊 상위 10개 품종:")
        for i, (breed, count) in enumerate(breed_counts.head(10).items()):
            percentage = count / len(df) * 100
            print(f"   {i+1:2d}. {breed}: {count}개 ({percentage:.1f}%)")
        
        # 품종별 severity 분포
        if 'severity' in df.columns:
            print(f"\n📊 품종별 평균 심각도 TOP 10:")
            breed_severity = df.groupby('dog_type')['severity'].agg(['mean', 'count']).sort_values('mean', ascending=False)
            breed_severity = breed_severity[breed_severity['count'] >= 10]  # 10마리 이상인 품종만
            
            for breed, row in breed_severity.head(10).iterrows():
                print(f"   {breed[:20]:20s}: {row['mean']:.2f} (n={row['count']:2d})")
    
    # 11. 요약 및 권장사항
    print(f"\n💡 ===== 분석 요약 및 권장사항 =====")
    
    print(f"✅ 전처리 품질:")
    quality_score = 0
    
    # 결측값 처리
    if len(missing_cols) == 0:
        print(f"   ✅ 결측값 완전 처리 (+20점)")
        quality_score += 20
    else:
        print(f"   ⚠️ 결측값 {len(missing_cols)}개 컬럼 존재 (+10점)")
        quality_score += 10
    
    # 특성 수
    if len(df.columns) > 50:
        print(f"   ✅ 풍부한 특성 {len(df.columns)}개 (+20점)")
        quality_score += 20
    else:
        print(f"   ⚠️ 특성 수 부족 {len(df.columns)}개 (+10점)")
        quality_score += 10
    
    # 클래스 분포
    if 'severity' in df.columns:
        if imbalance_ratio < 10:
            print(f"   ✅ 적절한 클래스 분포 (불균형 {imbalance_ratio:.1f}:1) (+20점)")
            quality_score += 20
        elif imbalance_ratio < 30:
            print(f"   ⚠️ 중간 클래스 불균형 ({imbalance_ratio:.1f}:1) (+15점)")
            quality_score += 15
        else:
            print(f"   ❌ 심각한 클래스 불균형 ({imbalance_ratio:.1f}:1) (+5점)")
            quality_score += 5
    
    # 데이터 크기
    if len(df) > 1000:
        print(f"   ✅ 충분한 데이터 크기 {len(df)}개 (+20점)")
        quality_score += 20
    else:
        print(f"   ⚠️ 데이터 크기 부족 {len(df)}개 (+10점)")
        quality_score += 10
    
    # 특성 다양성
    if len(pose_cols) > 10 and len(sensor_cols) > 5:
        print(f"   ✅ 다양한 특성 유형 (포즈+센서+의료) (+20점)")
        quality_score += 20
    else:
        print(f"   ⚠️ 제한적 특성 유형 (+10점)")
        quality_score += 10
    
    print(f"\n🎯 전처리 품질 점수: {quality_score}/100점")
    
    if quality_score >= 80:
        print(f"🎉 우수! 바로 모델링 진행 가능")
    elif quality_score >= 60:
        print(f"👍 양호! 일부 개선 후 모델링 권장")
    else:
        print(f"⚠️ 개선 필요! 추가 전처리 권장")
    
    print(f"\n🚀 다음 단계 권장사항:")
    print(f"   1. 클래스 불균형 처리 (SMOTE, 가중치 조정)")
    print(f"   2. 특성 선택 (상관관계 높은 상위 50개)")
    print(f"   3. 교차검증으로 모델 성능 평가")
    print(f"   4. 의료진 해석 가능한 특성 우선 사용")
    
    return df

# ===== 실행 =====
def main():
    """메인 분석 함수"""
    print("🔍 전처리된 데이터 분석을 시작합니다...")
    
    # 파일 경로 (저장한 CSV 파일명에 맞게 수정)
    csv_files = ['fixed_pet_data_1050.csv', 'processed_pet_data.csv', 'pet_health_processed.csv']
    
    df = None
    for csv_file in csv_files:
        try:
            df = analyze_preprocessed_data(csv_file)
            if df is not None:
                break
        except:
            continue
    
    if df is None:
        print("❌ CSV 파일을 찾을 수 없습니다.")
        print("다음 중 하나의 파일명으로 저장했는지 확인해주세요:")
        for file in csv_files:
            print(f"   - {file}")
    else:
        print(f"\n🎉 분석 완료! 모델링 준비 되었습니다!")

if __name__ == "__main__":
    main()

🔍 전처리된 데이터 분석을 시작합니다...
📊 ===== 전처리 결과 분석 시작 =====
✅ 데이터 로드 성공: (1050, 91)

📋 ===== 기본 정보 =====
📊 데이터 형태: (1050, 91)
📈 총 특성 수: 91
🎯 타겟 변수: severity
🔍 메모리 사용량: 0.85 MB

📂 ===== 특성 분류 =====
📋 기본 정보 컬럼: 6개
   ['pet_id', 'severity', 'age', 'dog_type', 'size']...
🦴 포즈 관련 컬럼: 28개
   ['P0_x', 'P0_y', 'P0_label', 'P0_conf', 'P1_x']...
📱 센서 관련 컬럼: 18개
   ['sensor_mean', 'sensor_std', 'sensor_min', 'sensor_max', 'sensor_median']...
🏥 의료 기록 컬럼: 4개
   ['medical_foot_position_0', 'medical_value_0', 'medical_foot_position_1', 'medical_value_1']
⚙️ 엔지니어링 컬럼: 11개
   ['timestamp_normalized', 'sensor_range', 'pose_confidence_mean', 'pose_confidence_std', 'sensor_instability']...

🎯 ===== 타겟 변수 분석 =====
📊 심각도 분포:
   Level 0: 596개 ( 56.8%) ████████████████████████████
   Level 1:  89개 (  8.5%) ████
   Level 2: 142개 ( 13.5%) ██████
   Level 3: 197개 ( 18.8%) █████████
   Level 4:  26개 (  2.5%) █
📈 클래스 불균형 비율: 22.9:1
🔄 이진분류 분포: 정상 596개 vs 질병 454개 (43.2%)

🔍 ===== 결측값 분석 =====
❌ 결측값이 있는 컬럼: 29개
   P12_y: 1030

In [14]:
# 분석 완료 코드
import pandas as pd
import numpy as np

def complete_analysis():
    """중단된 분석 완료"""
    
    # 데이터 로드
    df = pd.read_csv('fixed_pet_data_1050.csv')
    
    print("🔍 ===== 분석 완료 =====")
    
    # 1. 중요한 특성들 상세 분석
    print("\n⭐ ===== 핵심 특성 분석 =====")
    
    # medical_value들이 왜 이렇게 높은 상관관계를 가지는지 확인
    medical_cols = [col for col in df.columns if 'medical_value' in col]
    
    for col in medical_cols:
        print(f"\n📊 {col} 분석:")
        print(f"   - 고유값: {df[col].unique()}")
        print(f"   - Severity별 평균:")
        for sev in sorted(df['severity'].unique()):
            avg_val = df[df['severity'] == sev][col].mean()
            print(f"     Level {sev}: {avg_val:.2f}")
    
    # 2. 결측값 처리 전략 제안
    print(f"\n🔧 ===== 결측값 처리 전략 =====")
    
    missing_info = df.isnull().sum()
    high_missing = missing_info[missing_info > len(df) * 0.5]
    
    print(f"📊 50% 이상 결측인 특성들:")
    for col, missing_count in high_missing.items():
        missing_pct = missing_count / len(df) * 100
        print(f"   {col}: {missing_pct:.1f}% 결측")
        
        if 'P12' in col:
            print(f"     → P12는 제거 권장 (98% 결측)")
        elif 'P11' in col:
            print(f"     → P11은 플래그 변환 권장 (53% 결측)")
        elif 'P10' in col:
            print(f"     → P10은 보간 가능 (44% 결측)")
    
    # 3. 모델링 준비도 평가
    print(f"\n🎯 ===== 모델링 준비도 평가 =====")
    
    quality_score = 0
    
    # 데이터 크기 (20점)
    if len(df) >= 1000:
        print(f"   ✅ 충분한 데이터 크기: {len(df)}개 (+20점)")
        quality_score += 20
    
    # 특성 수 (20점)  
    if len(df.columns) >= 80:
        print(f"   ✅ 풍부한 특성 수: {len(df.columns)}개 (+20점)")
        quality_score += 20
        
    # 강한 예측 특성 존재 (30점)
    corr_with_target = df.select_dtypes(include=[np.number]).corrwith(df['severity']).abs()
    strong_features = corr_with_target[corr_with_target > 0.8]
    if len(strong_features) > 0:
        print(f"   ✅ 강한 예측 특성 존재: {len(strong_features)}개 (+30점)")
        quality_score += 30
    else:
        quality_score += 10
    
    # 클래스 불균형 (15점)
    severity_dist = df['severity'].value_counts()
    imbalance_ratio = severity_dist.max() / severity_dist.min()
    if imbalance_ratio < 30:
        print(f"   ✅ 적절한 클래스 불균형: {imbalance_ratio:.1f}:1 (+15점)")
        quality_score += 15
    else:
        print(f"   ⚠️ 심한 클래스 불균형: {imbalance_ratio:.1f}:1 (+5점)")
        quality_score += 5
    
    # 다양한 특성 유형 (15점)
    pose_cols = len([col for col in df.columns if any(f'P{i}_' in col for i in range(7))])
    sensor_cols = len([col for col in df.columns if 'sensor' in col])
    
    if pose_cols > 15 and sensor_cols > 10:
        print(f"   ✅ 다양한 특성 유형: 포즈({pose_cols}), 센서({sensor_cols}) (+15점)")
        quality_score += 15
    else:
        quality_score += 10
    
    print(f"\n🏆 최종 품질 점수: {quality_score}/100점")
    
    # 4. 맞춤형 권장사항
    print(f"\n💡 ===== 맞춤형 권장사항 =====")
    
    if quality_score >= 85:
        print(f"🎉 우수한 품질! 바로 모델링 시작 가능")
        print(f"   추천 모델: XGBoost, LightGBM (불균형 데이터에 강함)")
        print(f"   우선 특성: medical_value_0, medical_value_1")
        
    elif quality_score >= 70:
        print(f"👍 양호한 품질! 일부 개선 후 모델링")
        print(f"   1. P12 특성 제거")
        print(f"   2. SMOTE로 클래스 불균형 처리") 
        print(f"   3. 상위 50개 특성만 선택")
        
    else:
        print(f"⚠️ 개선 필요! 추가 전처리 권장")
    
    # 5. 즉시 시작 가능한 간단 모델링 코드 제공
    print(f"\n🚀 ===== 즉시 시작 가능한 모델링 =====")
    
    # 상관관계 높은 상위 특성들
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'severity']
    
    correlations = df[numeric_cols + ['severity']].corrwith(df['severity']).abs().sort_values(ascending=False)
    top_features = correlations.head(15).index.tolist()
    
    print(f"📊 추천 시작 특성 (상관관계 상위 15개):")
    for i, feature in enumerate(top_features):
        corr_val = correlations[feature]
        print(f"   {i+1:2d}. {feature[:35]:35s}: {corr_val:.3f}")
    
    print(f"\n💻 간단 모델링 코드:")
    print(f"""
# 기본 모델링 시작 코드
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

# 상위 특성만 사용
top_features = {top_features[:10]}
X_simple = df[top_features]
y = df['severity']

# 결측값 처리
X_simple = X_simple.fillna(X_simple.median())

# 모델 훈련
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
scores = cross_val_score(rf, X_simple, y, cv=5, scoring='f1_weighted')

print(f'평균 F1 점수: {{scores.mean():.3f}} (+/- {{scores.std() * 2:.3f}})')
""")
    
    return df, quality_score, top_features

# 실행
df, score, features = complete_analysis()
print(f"\n✅ 분석 완료! 품질점수: {score}점")

🔍 ===== 분석 완료 =====

⭐ ===== 핵심 특성 분석 =====

📊 medical_value_0 분석:
   - 고유값: [1. 0. 3. 2. 4.]
   - Severity별 평균:
     Level 0: 0.00
     Level 1: 0.76
     Level 2: 1.72
     Level 3: 2.67
     Level 4: 3.15

📊 medical_value_1 분석:
   - 고유값: [1. 0. 3. 4. 2.]
   - Severity별 평균:
     Level 0: 0.00
     Level 1: 0.69
     Level 2: 1.45
     Level 3: 2.54
     Level 4: 3.42

🔧 ===== 결측값 처리 전략 =====
📊 50% 이상 결측인 특성들:
   P11_x: 53.3% 결측
     → P11은 플래그 변환 권장 (53% 결측)
   P11_y: 53.3% 결측
     → P11은 플래그 변환 권장 (53% 결측)
   P11_conf: 53.3% 결측
     → P11은 플래그 변환 권장 (53% 결측)
   P12_x: 98.1% 결측
     → P12는 제거 권장 (98% 결측)
   P12_y: 98.1% 결측
     → P12는 제거 권장 (98% 결측)
   P12_conf: 98.1% 결측
     → P12는 제거 권장 (98% 결측)

🎯 ===== 모델링 준비도 평가 =====
   ✅ 충분한 데이터 크기: 1050개 (+20점)
   ✅ 풍부한 특성 수: 91개 (+20점)
   ✅ 강한 예측 특성 존재: 3개 (+30점)
   ✅ 적절한 클래스 불균형: 22.9:1 (+15점)
   ✅ 다양한 특성 유형: 포즈(28), 센서(18) (+15점)

🏆 최종 품질 점수: 100/100점

💡 ===== 맞춤형 권장사항 =====
🎉 우수한 품질! 바로 모델링 시작 가능
   추천 모델: XGBoost, LightGBM (불균형 데이터에 강함)
